***
# Mapping Wildfires
#### by Paul Ghisletti, as part of the SDS210 course (spring semester '26)

***

In [163]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os
import requests
from datetime import datetime
import unicodedata
import re
import sys
from pathlib import Path
import folium as fm
from folium import plugins
import branca.colormap as cm
from sklearn.cluster import DBSCAN

In [164]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In order to import packages from the `src/` folder, we need to set the root of this notebook to the repository's root folder:

In [165]:
sys.path.append(str(Path().resolve().parent))

***
## API-Setup
Make sure you followed the steps 2.1. to 2.3. in the `README.md` file on how to set your individual API-key: Check, whether the following output matches your individual API-key from FIRMS. Here, you can also see how many free transactions you have left with your api-key

In [166]:
# load the content of the `.env` file
load_dotenv()

# assigns the api-keay which is specified in the `.env` file to variable
api_key = os.getenv("MY_API_KEY")
print(f"API-Key: {api_key}")

# uses FIRMS api to get information on transactions
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + api_key
try:
  df = pd.Series(requests.get(url).json()) # gets a table of information about your api key usage
  display(df)
except:
  print ("There is an issue with your API key. Please check the value of MY_API_KEY in your .env file and your internet connection and try again.")

API-Key: 3d1ef73c3c932e5f3738d87f205a9cec


transaction_limit             5000
current_transactions           110
transaction_interval    10 minutes
dtype: object

We can also define a function which tells us how many transactions we have used so far. This can be used to check, how many transactions one API query costs (code from the [FIRMS notebook on *API use*](https://firms.modaps.eosdis.nasa.gov/academy/data_api/)):

In [167]:
def get_transaction_count() :
  count = 0
  url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + api_key
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print (f'Our current transaction count is {tcount}')

Our current transaction count is 110


***
## 1. Loading a dataset through the API
### DataClass for API output
By handling the API query input and output as a Class object, we can later easily access metadata or even define methods (e.g. .get_coordinates()). For this to work, we need to set up a DataClass:

In [168]:
from dataclasses import dataclass

@dataclass
class WildFireQuery:
    data: gpd.GeoDataFrame
    sensor: str
    area: str
    extent: str
    date: str
    n_days: int
    geometry: gpd.GeoDataFrame | None

    def __post_init__(self): # creates attributes after instantiation of object automatically
        # handles the special case of 'world'
        if self.area == "world":
            self.extent = "-180,-90,180,90"
        self.bbox = [float(n) for n in self.extent.split(",")]

        # display name, because it is lost during the api query
        if self.area == "world":
            self.area_display = "World"

        # we can use the gdf for continents and countries to look up the names again
        elif self.area in continents_gdf["CONTINENT_NORM"].values:
            match = continents_gdf[
                continents_gdf["CONTINENT_NORM"] == self.area
            ]

            self.area_display = (
                match["CONTINENT"].values[0]
                if not match.empty
                else self.area
            )

        else:
            match = countries_gdf[
                countries_gdf["name"] == self.area
            ]

            self.area_display = (
                match["ADMIN"].values[0]
                if not match.empty
                else self.area
            )

    # calculate coordinates of polygon center for centering the final folium map
    def get_center(self) -> tuple:
        if self.area == "world":
            return (20, 0)
        bbox = self.bbox
        return ((bbox[1] + bbox[3]) / 2,
                (bbox[0] + bbox[2]) / 2)
    
    # calculate fitting start zoom based on area extent for final folium map
    def get_zoom_level(self) -> int:
        bbox = self.bbox
        max_extent = max(bbox[2] - bbox[0],
                         bbox[3] - bbox[1])

        # thresholds were picked manually 
        if max_extent >= 180: return 2
        elif max_extent > 60: return 3
        elif max_extent > 30: return 4
        elif max_extent > 20: return 5
        elif max_extent > 10: return 6
        elif max_extent > 5:  return 7
        else:                 return 8

### 1.1. Check availability
Because the latency differs between the sensors (actually, it mostly changes between different products of the same sensor provided by FIRMS. The -SP products have been processed by FIRMS extensively and contain the extra variabla `type`. This takes some time, which is why the latency for those outputs is usually a couple of months, whereas the latency for -NRT products is almost instantaneous), we first need to check the date availability. We can later use this information when automating the download.  

This function is important for later use, where we want to automate the process of getting data from the API. Since the user usually does not know the date of the latest available data, we need to look this up automatically, which is where this function comes into play.

In [169]:
def get_availability_all(api_key):
    url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + api_key + '/ALL'
    return pd.read_csv(url) 

availability_all_df = get_availability_all(api_key)
display(availability_all_df.head(10))

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-19
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-19
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-19
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-19
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-19
8,GOES_NRT,2022-08-09,2026-05-19
9,BA_MODIS,2000-11-01,2026-02-01


### 1.2. Download data
In this chapter, we will define a function which will automatically get the API data. All the user has to do is provide the following parameters:
- The desired sensor
- The area which the data should cover
- The date of data acquisition
- Time frame of the data  

Since this is a function where we need to check numerous parameter inputs, it is helpful to define helper functions to make the final function more readable. These helpers mainly check for correct input format and type.  

Because we enable the user to simply type the continent or country for the desired extent, we need a dataset that stores the bounding box coordinates which we can reference the user input against. This is done by importing a `.gpkg` file with all 198 countries of the world (193 UN members, 2 UN observer states, 3 disputed states) and one for the 7 continents. 
First, we import the `.gpkg` files as a `GeoDataFrame`, which we created from `ne_10m_admin_0_countries.shp` and `continent_boundaries_7.gpkg`respectively in the dedicated Jupyter Notebook (`/notebooks/countries_normalisation.ipynb`).

In [170]:
countries_gdf = gpd.read_file("../data/raw/198_countries.gpkg").to_crs(epsg=4326)

continents_gdf = gpd.read_file("../data/raw/7_continents.gpkg").to_crs(epsg=4326)

The following chunk of code defines all the helper functions needed for the main function and the main function itself. Since there is a lot going on, I will state the steps performed by the function in words below:
- It checks for each input parameter, whether it is a valid input by the user, raising a `ValueError` otherwise.
    - Sensor: Must be one of the available sensors from the FIRMS API.
    - Area: explained below.
    - Date: valid string format: YYYY-MM-DD and it is within the provided range of dates by the FIRMS API.
    - N_days: at least 1 but only up to 5 days in one query.
- It uses the `ALIASES` dictionary from `/data/raw` (which was defined by generative AI) in a normalisation function to map user input to a valid country name. This accepts a wide range of alternative spellings and handles special characters, empty spaces and punctuation. It is worth noting that this also works with continents and "world".
- Using the `countries_gdf` or `continents_gdf` from above, which contain the normalised country and continent names as well as their geometry, it gets the bounding coordinates of the query extent. For continents it uses a dictionary where the 7 coordinate lists are predefined.
- Handling the date is also non-trivial, since the user can leave this parameter empty if they wish to get the latest data. Therefore, we need to consult the data frame which we fetched from the FIRMS API a couple steps above. This data frame contains the latest available date for each of the sensors. My function then either uses this date or the date provided by the user, given it is older than the latest possible date, raising a `ValueError` otherwise.
- It then constructs the API-URL which is how we pass all the input information to the FIRMS API.
- With the URL we can get the data which is directly converted from a `pd.DataFrame` to a `gpd.GeoDataFrame` with the `lat` and `lon` columns.

- Before finalising the output, we clip the data points to the geometry of the data extent defined by the user (e.g. a country or an entire continent). This prevents confusing outputs when looking for wildfires data in France, since the bounding box of the France geometry basically covers the entire globe with all its overseas territories.
- Both the newly calculated output and the user input is then passed into a new `WildFireQuery` class object.

In [171]:
# -----------------------------------------------
# Helper Functions for checking parameter inputs:
# -----------------------------------------------
def check_sensor(sensor):
    valid_sensors = set(availability_all_df["data_id"]) 
    if sensor not in valid_sensors:
        raise ValueError("Sensor: Invalid sensor name.")

def check_area_list(area):
    west, south, east, north = area
    # check invalid longitude values
    if west < -180 or west > 180: 
        raise ValueError("Area: Invalid coordinate: west")
    if east < -180 or east > 180: 
        raise ValueError("Area: Invalid coordinate: east")
    # check invalid latitude values
    if south < -90 or south > 90:
        raise ValueError("Area: Invalid coordinate: south")
    if north < -90 or north > 90:
        raise ValueError("Area: Invalid coordinate: north")
    # check order of list
    if west > east:
        raise ValueError("Area: Invalid coordinate: west is larger than east")
    if south > north:
        raise ValueError("Area: Invalid coordinate: south is larger than north")
    
def check_date_str(date, sensor, max_date_time):
    is_none = date is None
    is_str = type(date) == str

    # get 
    if not is_none and not is_str:
        raise ValueError("Date: Invalid input type. Either str or None")
    
    if is_str:
        try:
            date_time = datetime.strptime(date, "%Y-%m-%d")
        except ValueError:
            raise ValueError("Date: Must be in YYYY-MM-DD format and be a valid calendar date")
        if date_time > max_date_time:
            raise ValueError(f"Date: Desired date not available for {sensor}. Latest available data from {max_date_time}")
        
def check_n_days(n_days):
    if not type(n_days) == int:
        raise ValueError("N_days: Type must be int.")
    if n_days < 1 or n_days > 5:
        raise ValueError("N_days: Minimum 1 day, maximum 5 days.")

# --------------------------------------------------------------------
# Helper Functions for calculating area based on country or continent:
# --------------------------------------------------------------------

# import aliases dictionary from src folder
from src.country_continent_aliases import ALIASES

# function for normalising a country string
def normalise(country: str) -> str:
    country = unicodedata.normalize("NFD", country) # decomposes special characters: ô → o + ^
    country = "".join(c for c in country if unicodedata.category(c) != "Mn") # drop all accents (diacritic characters)
    country = country.lower() # lowercase everything
    country = re.sub(r"[^a-z0-9\s]", "", country) # removes forbidden characters
    country = re.sub(r"\s+", "", country) # removes multi-spaces in middle and all spaces at beginning and end
    country = ALIASES.get(country, country)
    return country

def is_country(area: str) -> bool:
    """
    Checks, whether area input is a continent
    """
    is_country = (area in countries_gdf["name"].values
                  or area.upper() in countries_gdf["ISO_A3"].values)
    return is_country

def is_continent(area: str) -> bool:
    """
    Checks, whether area input is a continent
    """
    is_continent = area in continents_gdf["CONTINENT_NORM"].values
    return is_continent

def is_world(area: str) -> bool:
    """
    Checks, whether area input is equal to 'world'
    """
    return area == "world"
    
# get geometry for either continent or country
def get_geometry(area: str) -> gpd.GeoDataFrame | None:
    """
    Gets a GeoDataFrame with the geometry of the input area. For 'world', no geometry is needed.
    """
    if is_country(area):
        geom = countries_gdf[(countries_gdf["name"] == area) |
                             (countries_gdf["ISO_A3"] == area.upper())]
    if is_continent(area):
        geom = continents_gdf[continents_gdf["CONTINENT_NORM"] == area]
    if is_world(area):
        geom = None
    return geom
    

# return list of 4 coordinates for input country
def get_area_coord(area: str) -> list:
    """
    Gets the bounding box coordinates for any valid user input.
    """
    # normalise input using ALIASES dictionary
    area = normalise(area)

    # check validity of area input
    if not is_continent(area) and not is_country(area):
        raise ValueError("Area: Invalid country code, country name or continent.")
    
    # retrieve bbox for continets if input valid
    if is_continent(area):
        continent_gdf = get_geometry(area)
        minx, miny, maxx, maxy = continent_gdf.geometry.total_bounds
        bbox = [minx, miny, maxx, maxy]

    # retrieve bbox for countries if input valid
    else:
        country_gdf = get_geometry(area)
        minx, miny, maxx, maxy = country_gdf.geometry.total_bounds
        bbox = [minx, miny, maxx, maxy]
    print(f"Area name normalised: {area}")
    return [area, ",".join(map(str, bbox))]

# clip data points to country/continent extent
def clip(output_gdf: gpd.GeoDataFrame, area: str) -> gpd.GeoDataFrame:
    """
    Clips the gdf output from the API to the user input area geometry
    """
    area = normalise(area)
    if is_country(area):
        clipped_gdf = gpd.clip(output_gdf, countries_gdf[countries_gdf["name"] == area])
    if is_continent(area):
        clipped_gdf = gpd.clip(output_gdf, continents_gdf[continents_gdf["CONTINENT_NORM"] == area])
    return clipped_gdf

# ----------------------------------------
# Helper function for converting df to gdf
# ----------------------------------------
def df_to_gdf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """
    Converts the pd.DataFrame output from API query to gpd.GeoDataFrame.
    """
    gdf = gpd.GeoDataFrame(
        data=df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs="EPSG:4326"
    )
    return gdf

# ------------
# API Function
# ------------
def area_api_query(sensor: str, area: list|str = 'world', date: str | None = None, n_days: int = 1) -> pd.DataFrame:
    """
    Query the NASA FIRMS API for area data for a given sensor and date. If date is not provided, it will return the most recent data.

    Choose one of these sensors:
        'LANDSAT_NRT',
        'MODIS_NRT',
        'MODIS_SP',
        'VIIRS_NOAA20_NRT',
        'VIIRS_NOAA20_SP',
        'VIIRS_NOAA21_NRT',
        'VIIRS_SNPP_NRT',
        'VIIRS_SNPP_SP'

    Parameters:
    -----------
    sensor : str
        Must be from the sensors list.

    area : list or str, optional
        The area for which to query data, 4 options:
        - 'world' for global coverage
        - Country name or ISO 3166-1 alpha-3 country codes (e.g. "Algeria" or "DZA")
        - Continent name (e.g. "North America")
        - Coordinate bounding box with format [west,south,east,north] (e.g. [-180,-90,180,90])

    date : str, optional
        Date in the format 'YYYY-MM-DD'. If not provided, the most recent data will be returned.

    n_days: int, optional
        Number of days that the data should cover.
        Max = 5
        Default value = 1

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the area data for the specified sensor and date.
    """
    # sensor
    # ------
    if check_sensor(sensor) == False:
        raise ValueError("Sensor: Invalid sensor name, check list of available sensors.")
    print(f"Sensor input: {sensor}")
    
    # area
    # ----
    print(f"Area input: {area}")
    if type(area) == str:
        if is_world(area):
            area_norm = 'world'
            extent= area
        else:
            area_list = get_area_coord(area)
            area_norm = area_list[0]
            extent = area_list[1]
    elif type(area) == list:
        check_area_list(area)
        extent = ",".join(map(str, area)) # converts input list to str for url use
    else:
        raise ValueError("Area: Anvalid input type. Must be either str or list")
    print(f"Area bbox: {extent}")

    # date
    # ----
    availability_all_df = get_availability_all(api_key)
    max_date = availability_all_df.loc[
            availability_all_df["data_id"] == sensor,
            "max_date"
        ].iloc[0]
    max_date_time = datetime.strptime(max_date, "%Y-%m-%d")

    check_date_str(date, sensor, max_date_time)

    if date is None:
        date_str = max_date_time.strftime("%Y-%m-%d")
    if type(date) == str:
        date_str = date
    print(f"Date input: {date_str}")

    # n_days
    # ------
    check_n_days(n_days)
    n_days = str(n_days)
    print(f"N_days input: {n_days}")
    print("All input correct")

    # API query
    # ---------
    area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + api_key + '/' + sensor + '/' + extent + '/' + n_days + '/' + date_str
    print(f"URL: {area_url}")
    df = pd.read_csv(area_url)

    # Convert to gpd.GeoDataFrame
    unclipped_gdf = df_to_gdf(df)

    # clip to area boundaries
    if is_world(area):
        output_gdf = unclipped_gdf
    else:
        output_gdf = clip(unclipped_gdf, area)
    print(f"Shape (rows, columns): {output_gdf.shape}")

    # iinstantiate WildFireQuery object
    output = WildFireQuery(
        data = output_gdf,
        sensor = sensor,
        area = area_norm,
        extent = extent,
        date = date_str,
        n_days = n_days,
        geometry= get_geometry(normalise(area))
    )
    print (f'Our current transaction count is {get_transaction_count()}/5000')
    return output

Let us check, whether this monstrosity of a function actually works:

In [184]:
wf = area_api_query(
    'MODIS_SP',
    n_days = 5,
    area="car",
    date= '2026-02-28'	
    )
display(wf.data.head(1))
print(wf.extent)

Sensor input: MODIS_SP
Area input: car
Area name normalised: centralafricanrepublic
Area bbox: 14.387266072000074,2.2364537560000457,27.441301310000142,11.000828349000074
Date input: 2026-02-28
N_days input: 5
All input correct
URL: https://firms.modaps.eosdis.nasa.gov/api/area/csv/3d1ef73c3c932e5f3738d87f205a9cec/MODIS_SP/14.387266072000074,2.2364537560000457,27.441301310000142,11.000828349000074/5/2026-02-28
Shape (rows, columns): (2460, 16)
Our current transaction count is 85/5000


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type,geometry
665,3.6698,18.2076,309.2,1.5,1.2,2026-02-28,1342,Aqua,MODIS,46,61.03,293.1,8.9,D,0,POINT (18.2076 3.6698)


14.387266072000074,2.2364537560000457,27.441301310000142,11.000828349000074


## 2. Cleaning the dataframe
Before trying to visualise all the interesting data contained in the dataset, we must clean it first and prepare it for future steps. We can get an overview of the dataset by looking at the columns and what type of data they contain.



In [185]:
def overview(df, name="dataset"):
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(dropna=True)
    }).sort_values(["dtype", "n_unique"], ascending=[True, False])

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    return summary

overview(wf.data)


=== dataset ===
Shape: (2460, 16)


,dtype,n_unique
longitude,float64,2425
latitude,float64,2379
frp,float64,532
brightness,float64,506
bright_t31,float64,340
scan,float64,25
track,float64,9
version,float64,1
geometry,geometry,2460
confidence,int64,94


Based on these insights, we can now decide how to clean and change the variables and entries:
- change the acquisition date and acquisition time to a valid datetime format.  

It seems like the API delivers an already clean data set which is why no more steps are needed.
### 2.1. Cleaning Function
At the moment, the time and date are stored in separate columns as `str`. The following step will convert it into a single column containing gpd.datetime objects. Afterwards, we can drop the now useless `acq_time` and `acq_date` columns.

In [186]:
def clean(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Cleans a raw wildfire GeoDataFrame by removing invalid rows.
    Drops rows silently and prints a summary of what was removed.
    - Converts acq_time and acq_date to datetime
    - Checks coordinate plausibility
    - Drops rows with frp < 0
    - Normalises frp to MW per km^2
    - Checks brightness plausibility
    - Checks confidence plausibility
    - Removes low confidence entries
    - Checks Day/Night
    - Drops duplicate rows

    Parameters:
    -----------
    gdf : gpd.GeoDataFrame
        Raw GeoDataFrame from FIRMS API query

    Returns:
    --------
    gpd.GeoDataFrame
        Cleaned GeoDataFrame
    """
    original_len = len(gdf)
    dropped = {}

    # Datetime
    if "acq_time" in gdf.columns:
        gdf["datetime"] = pd.to_datetime(
            gdf["acq_date"] + " " + gdf["acq_time"].astype(str).str.zfill(4),
            format="%Y-%m-%d %H%M",
            utc=True
        )
        gdf = gdf.drop(columns=["acq_time", "acq_date"])

    # Coordinates
    mask = (
        gdf["latitude"].between(-90, 90) &
        gdf["longitude"].between(-180, 180)
    )
    dropped["invalid_coordinates"] = (~mask).sum()
    gdf = gdf[mask]

    # FRP cleaninf
    if "frp" in gdf.columns:
        mask = gdf["frp"] > 0
        dropped["invalid_frp"] = (~mask).sum()
        gdf = gdf[mask]

    # FRP normalisation
    if "frp" in gdf.columns:
        gdf["frp_density"] = (gdf["frp"] / (gdf["scan"] * gdf["track"])).round(1) # MW per km^2

    # Brightness temperatures
    bt_checks = {
        "brightness" : (200, 800),
        "bright_t31" : (200, 400),
        "bright_ti4" : (200, 800),
        "bright_ti5" : (200, 400),
    }
    dropped["invalid_brightness"] = 0
    for col, (low, high) in bt_checks.items():
        if col in gdf.columns:
            mask = gdf[col].between(low, high)
            dropped["invalid_brightness"] += (~mask).sum()
            gdf = gdf[mask]

    # Confidence
    # MODIS
    if pd.api.types.is_numeric_dtype(gdf["confidence"]):
        # remove invalid confidence
        mask_valid = gdf["confidence"].between(0, 100)
        dropped["invalid_confidence"] = (~mask_valid).sum()
        gdf = gdf[mask_valid]
        # remove unconfident entries: > 30%
        mask_likely = gdf["confidence"] > 30
        dropped["low_confidence"] = (~mask_likely).sum()
        gdf = gdf[mask_likely]
    # VIIRS
    else:
        # remove unconfident entries: "l" (= low)
        dropped["invalid_confidence"] = 0
        mask_likely = gdf["confidence"].isin(["n", "h"])
        dropped["low_confidence"] = (~mask_likely).sum()
        gdf = gdf[mask_likely]

    # Day/Night flag
    mask = gdf["daynight"].isin(["D", "N"])
    dropped["invalid_daynight"] = (~mask).sum()
    gdf = gdf[mask]

    # Duplicates
    duplicate_mask = gdf.duplicated(subset=["latitude", "longitude", "datetime", "satellite"])
    dropped["duplicates"] = duplicate_mask.sum()
    gdf = gdf[~duplicate_mask]

    # Summary
    total_dropped = original_len - len(gdf)
    print(f"=== Cleaning Summary ===")
    print(f"Invalid coordinates  : {dropped['invalid_coordinates']}")
    print(f"Invalid FRP          : {dropped['invalid_frp']}")
    print(f"Invalid brightness   : {dropped['invalid_brightness']}")
    print(f"Invalid confidence   : {dropped['invalid_confidence']}")
    print(f"Low confidence       : {dropped["low_confidence"]}")
    print(f"Invalid daynight     : {dropped['invalid_daynight']}")
    print(f"Duplicates           : {dropped['duplicates']}")
    print(f"-------------------------")
    print(f"Rows removed: {total_dropped} of {original_len}")
    print(f"Rows remaining: {len(gdf)}")

    return gdf.reset_index(drop=True)

In [187]:
wf.data = clean(wf.data)

=== Cleaning Summary ===
Invalid coordinates  : 0
Invalid FRP          : 0
Invalid brightness   : 0
Invalid confidence   : 0
Low confidence       : 144
Invalid daynight     : 0
Duplicates           : 0
-------------------------
Rows removed: 144 of 2460
Rows remaining: 2316


## 3. Interactive Map
Now we can tackle the heart of the project: The visualisation. The plan is to create a singular interactive map with folium and then let the user toggle each layer to their liking.
### 3.1. Clustering
If we want to find especially large fires, we need to cluster single fire pixels together. The best approach for this is to use DBSCAN, since it handles noise well and we don't have to predefine the number of clusters. The two parameters we can define are `min_samples` (the minimum number of fire pixels in one cluster, all other points are not assigned to any cluster), and `eps` (epsilon: maximum Distance from nearest neighbour in a cluster). In our case, an epsilon lower than the pixel size of the sensor would result in practically no clusters, since the fire pixels are by design of the sensor at least one pixel size apart. But if we make epsilon too large, we could risk grouping several fires into one.

In [188]:
KM_PER_RADIAN = 6371.0088 # used, because Haversine uses radians instead of degrees

# different epsilon values for different sensor resolutions
if wf.data["instrument"].mode()[0] == "MODIS":
    epsilon_km = 2
elif wf.data["instrument"].mode()[0] == "VIIRS":
    epsilon_km = 0.7
elif wf.data["instrument"].mode()[0] == "LANDSAT":
    epsilon_km = 0.05

if 'type' in wf.data.columns and wf.data['type'].notna().any(): # checks, whether the API data contains 'type' and whether it contains any NaN values
    all_clusters = []
    offset = 0 # ensures unique cluster IDs across all types
    
    for fire_type, group in wf.data.groupby('type'):
        # extract lon/lat for each fire pixel
        coords_rad = np.radians(
            np.column_stack([group.geometry.y, group.geometry.x])
        )
        # perform DBSCAN clustering
        db = DBSCAN(
            eps=epsilon_km / KM_PER_RADIAN,
            min_samples=2,
            algorithm='ball_tree', # required when using haversine
            metric='haversine' # handles lat, lon coordinates, as compared to using metric and having to reproject twice
        ).fit(coords_rad)
        
        # create labels which we can add to the initial gdf
        labels = db.labels_.copy()
        labels[labels >= 0] += offset  # avoid cluster ID collisions across types
        offset = labels.max() + 1 if labels.max() >= 0 else offset # updates offset so the cluster IDs of the next iteration start where the last iteration ended
        
        all_clusters.append(pd.Series(labels, index=group.index))
    
    wf.data['cluster'] = pd.concat(all_clusters) # creates a single series and adds it as a column to wf.data (our main gdf)

    clustered = wf.data[wf.data['cluster'] >= 0].copy()

else: # if we don't have type, just cluster over all points, same procedure
    coords_rad = np.radians(
        np.column_stack([wf.data.geometry.y, wf.data.geometry.x])
    )
    db = DBSCAN(
        eps=epsilon_km / KM_PER_RADIAN,
        min_samples=2,
        algorithm='ball_tree',
        metric='haversine'
    ).fit(coords_rad)
    wf.data['cluster'] = db.labels_

    clustered = wf.data[wf.data['cluster'] >= 0].copy()

brightness_cols = ['brightness', 'bright_t31', 'bright_ti4', 'bright_ti5']

agg_kwargs = {'pixel_count': ('geometry', 'count'),
              'frp_d_mean': ('frp_density', 'mean'),
              'frp_d_max': ('frp_density', 'max'),
              'first_pixel': ('datetime', 'min'),
              'last_pixel': ('datetime', 'max'),
              'time_mean': ('datetime', 'mean')}

# adds a brightness mean and max for all available brightness variables (e.g. bright_ti4, bright_t31)
for col in brightness_cols:
    if col in clustered.columns:
        agg_kwargs[f'{col}_max'] = (col, 'max')
        agg_kwargs[f'{col}_mean'] = (col, 'mean')

if 'type' in clustered.columns:
    agg_kwargs['type'] = ('type', 'first')

# uses the specific agg_kwargs to add stats to the clusters
stats = clustered.groupby('cluster').agg(**agg_kwargs)

# and rounds the values for readability
round_cols = ['frp_d_mean']
for col in brightness_cols: # goes through all contained brightness columns
    if col in stats.columns:
        round_cols.append(col)
stats['frp_d_mean'] = stats[round_cols].round(1)

# Key step: dissolving to get singular point per fire cluster
centroids = clustered.dissolve(by='cluster').centroid.rename('geometry') 

cluster_gdf = gpd.GeoDataFrame(stats, geometry=centroids, crs=wf.data.crs)
cluster_gdf["time_span"] = cluster_gdf["last_pixel"] - cluster_gdf["first_pixel"]
cluster_gdf = cluster_gdf.sort_values("pixel_count", ascending=False)
display(cluster_gdf.head(20))
overview(cluster_gdf)

/var/folders/br/770tdjgs0s11gdcqp7p99g_00000gn/T/ipykernel_65400/357521520.py:82: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = clustered.dissolve(by='cluster').centroid.rename('geometry')


,pixel_count,frp_d_mean,frp_d_max,first_pixel,last_pixel,time_mean,brightness_max,brightness_mean,bright_t31_max,bright_t31_mean,type,geometry,time_span
cluster,,,,,,,,,,,,,
369,23,49.1,139.9,2026-02-28 06:51:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 15:27:33.913043+00:00,381.5,341.991304,310.9,297.539130,0,POINT (23.86152 7.22792),0 days 12:23:00
367,19,41.0,192.1,2026-02-28 06:51:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 15:53:50.526315+00:00,391.9,336.047368,309.8,296.678947,0,POINT (24.1044 7.19025),0 days 12:23:00
426,14,44.6,150.8,2026-02-28 13:43:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 16:52:08.571428+00:00,381.0,335.321429,292.3,285.700000,0,POINT (21.23681 7.95079),0 days 05:31:00
365,13,10.5,21.2,2026-02-28 01:20:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 11:35:41.538461+00:00,332.4,317.869231,305.2,297.507692,0,POINT (23.85405 6.73974),0 days 17:54:00
374,12,15.3,44.0,2026-02-28 01:20:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 14:03:10+00:00,347.6,323.241667,306.3,299.608333,0,POINT (23.59642 6.75407),0 days 17:54:00
454,11,34.0,83.9,2026-02-28 13:43:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 15:13:16.363636+00:00,361.1,333.081818,288.1,279.390909,0,POINT (21.98768 8.34665),0 days 05:31:00
362,10,25.5,75.3,2026-02-28 13:42:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 14:48:24+00:00,360.2,330.860000,307.1,300.820000,0,POINT (24.67586 6.55934),0 days 05:32:00
219,10,134.0,616.4,2026-02-28 13:42:00+00:00,2026-02-28 13:42:00+00:00,2026-02-28 13:42:00+00:00,444.2,362.040000,326.0,306.690000,0,POINT (19.76203 5.88016),0 days 00:00:00
357,10,14.6,30.0,2026-02-28 13:42:00+00:00,2026-02-28 19:14:00+00:00,2026-02-28 14:15:12+00:00,339.6,326.740000,305.1,301.770000,0,POINT (23.94135 6.67634),0 days 05:32:00



=== dataset ===
Shape: (527, 13)


,dtype,n_unique
time_mean,"datetime64[us, UTC]",28
first_pixel,"datetime64[us, UTC]",6
last_pixel,"datetime64[us, UTC]",4
brightness_mean,float64,440
bright_t31_mean,float64,421
frp_d_max,float64,313
brightness_max,float64,299
frp_d_mean,float64,240
bright_t31_max,float64,225
geometry,geometry,527


In [189]:
overview(wf.data)


=== dataset ===
Shape: (2316, 17)


,dtype,n_unique
datetime,"datetime64[us, UTC]",10
longitude,float64,2284
latitude,float64,2241
frp,float64,527
brightness,float64,493
frp_density,float64,448
bright_t31,float64,336
scan,float64,25
track,float64,9
version,float64,1


### 3.2. Mapping

In [192]:
def map_wf(wf: WildFireQuery, countries_gdf: gpd.GeoDataFrame) -> fm.Map:
    """
    map_wf creates a folium map with wildfire data from FIRMS API

    Map layers:
    - heatmap

    Parameters:
    -----------
    gdf : gpd.GeoDataFrame
        Must be from the sensors list.

    area : list or str, optional
        The area for which to query data, 4 options:
        - 'world' for global coverage
        - Country name or ISO 3166-1 alpha-3 country codes (e.g. "Algeria" or "DZA")
        - Continent name (e.g. "North America")
        - Coordinate bounding box with format [west,south,east,north] (e.g. [-180,-90,180,90])

    date : str, optional
        Date in the format 'YYYY-MM-DD'. If not provided, the most recent data will be returned.

    n_days: int, optional
        Number of days that the data should cover.
        Max = 5
        Default value = 1

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the area data for the specified sensor and date.
    """
    # prepare clusters gdf for mapping in folium
    # ---------------------------------------------
    # convert datetime to strings for JSON handling
    cluster_gdf_plot = cluster_gdf.copy()
    # datetimes
    cluster_gdf_plot["first_pixel"] = cluster_gdf_plot["first_pixel"].dt.strftime("%Y-%m-%d %H:%M UTC")
    cluster_gdf_plot["last_pixel"] = cluster_gdf_plot["last_pixel"].dt.strftime("%Y-%m-%d %H:%M UTC")
    cluster_gdf_plot["time_mean"] = cluster_gdf_plot["time_mean"].dt.strftime("%Y-%m-%d %H:%M UTC")
    cluster_gdf_plot["time_span"] = cluster_gdf_plot["time_span"].apply(
        lambda x: f"{int(x.total_seconds() // 3600)}h {int((x.total_seconds() % 3600) // 60)}m" # lambda just used to construct the string and pass x
    )

    # convert type from ordinal numbers to readable names
    types_dict = {
        0: 'vegetation fire',
        1: 'active volcano',
        2: 'other (land)',
        3: 'offshore'
    }
    if 'type' in cluster_gdf_plot.columns:
        cluster_gdf_plot['type'] = cluster_gdf_plot['type'].apply(lambda x: types_dict[x])

    # add column to use later for tooltip: what kind of object am I looking at?
    cluster_gdf_plot["display_type"] = "Fire Cluster"

    # initiate the folium map canvas
    # ------------------------------
    center = wf.get_center()
    zoom_start = wf.get_zoom_level()
    m = fm.Map(
        location=center,
        zoom_start=zoom_start,
        min_zoom = zoom_start,
        control_scale=True,
        tiles= "CartoDB DarkMatter")
    
    # Add Data Information
    # --------------------
    title_html = f"""
    <div style="
        position: fixed;
        top: 10px;
        left: 50px;
        z-index: 1000;
        background-color: rgba(0,0,0,0.6);
        color: white;
        padding: 10px 15px;
        border-radius: 5px;
        font-family: Arial;
        font-size: 13px;
    ">
        <b>Wildfire Detections</b><br>
        Sensor: {wf.sensor}<br>
        Date: {wf.date}<br>
        Area: {wf.area_display}<br>
        Days: {wf.n_days}
    </div>
    """

    m.get_root().html.add_child(fm.Element(title_html))

    # area outline
    # ------------
    if wf.geometry is not None:
        outline_group = fm.FeatureGroup(name="Area outline", show=True)
        fm.GeoJson(wf.geometry,
                style_function=lambda x: {
                    "color": "white",
                    "weight": 1,
                    "opacity": 0.7,
                    "fillOpacity": 0
                    }
                ).add_to(outline_group)
        outline_group.add_to(m)

    # Heatmap
    # -------
    heat_data = wf.data[["latitude", "longitude", "frp_density"]].values.tolist()
    heat_group = fm.FeatureGroup(name="Heatmap", show=True)
    plugins.HeatMap(
        heat_data,
        min_opacity=0.4,
        radius=8,
        blur=6,
        max_zoom=10
        ).add_to(heat_group)
    heat_group.add_to(m)

    # fire clusters
    # -------------
    max_pixel_count = cluster_gdf_plot['pixel_count'].max()
    def style_fn(feature):
        pixel_count = feature['properties']['pixel_count']
        return {
            'radius': max(3, pixel_count ** 1),  # scale however you like
            'color': 'cyan',
            'weight': 1,
            'opacity': max(0.5, pixel_count/max_pixel_count)
        }
    fire_clusters = fm.FeatureGroup(name="Fires", show=False)
    fm.GeoJson(
        cluster_gdf_plot,
        marker=fm.CircleMarker(
            fill=True,
            fill_opacity=0,
            fill_color='cyan'
        ),
        tooltip=fm.GeoJsonTooltip(
            fields=["display_type", "pixel_count", "frp_d_mean", "frp_d_max", "time_span"],
            aliases=["Object:", "number of pixels in cluster", "mean FRP density:", "max FRP density:", "Time Span:"]
            ),
        style_function=style_fn,
    ).add_to(fire_clusters)
    fire_clusters.add_to(m)

    # pure detection points
    # ---------------------
    # Create colormap
    frp_min = wf.data["frp_density"].min()
    frp_max = wf.data["frp_density"].max()

    colors = cm.linear.YlOrRd_09.colors[::-1]

    colormap = cm.LinearColormap(
        colors=colors,
        vmin=frp_min,
        vmax=frp_max
    )
    colormap.caption = "Fire Radiative Power (FRP) per km^2"

    # plot
    points_group = fm.FeatureGroup(name="Fire Pixels by FRP density", show=False)
    gdf_plot = wf.data.copy()
    gdf_plot['display_type'] = 'Fire Pixel' # add column to use later for tooltip: what kind of object am I looking at?
    gdf_plot["datetime"] = gdf_plot["datetime"].dt.strftime("%Y-%m-%d %H:%M UTC")
    fm.GeoJson(
        gdf_plot,
        marker=fm.CircleMarker(radius=2,
        fill=True,
        fill_color="orange",
        fill_opacity=0.5,
        weight=0),
        style_function=lambda feature: {
            "color": colormap(feature["properties"]["frp"]),
            "fillColor": colormap(feature["properties"]["frp"]),
        },
        tooltip=fm.GeoJsonTooltip(
            fields=["display_type", "frp_density", "datetime"],
            aliases=["Object:", "FRP density:", "Time:"]
        )
    ).add_to(points_group)
    points_group.add_to(m)

    fm.LayerControl(collapsed=False).add_to(m)
    plugins.MeasureControl(
        position="bottomleft",
        primary_length_unit="kilometers",
        secondary_length_unit="miles",
        primary_area_unit="sqmeters",
        secondary_area_unit="acres",
    ).add_to(m)
    return m

In [193]:
m = map_wf(wf, countries_gdf)
m